
# CSE 4310 
# Programming Assignment 2 (P2)

Alex Nguyen
1002097417

### Task 1 (30 points)

Import the "peppers.jpg" image, convert it to grayscale and apply

<img src = Images/peppers.jpg>

a) Sobel edge detector (Apply horizontal and vertical sobel filters and display the Sobel magnitude image)

b) Prewitt edge detector (Apply horizontal and vertical prewitt filters and display the Prewitt magnitude image)

c) Laplacian edge detector 

d) Edge detection using difference of Gaussians (DoG)
    
Note: 
1. You can't use opencv or any other library function to detect edges
2. Apply zero-padding to make the filtered image size same as original image. Display the original image with the filtered images in a 1X2 plot for each edge detector. 

In [ ]:
import cv2 
import matplotlib.pyplot as plt
import numpy as np

In [ ]:

########################## Task 1A ###########################
imported_pepper_bgr = cv2.imread("Images/peppers.jpg")
grayscale_pepper = cv2.cvtColor(imported_pepper_bgr, cv2.COLOR_BGR2GRAY)

def manual_correlation(img, row, col, kernel):
    output = 0
    for x in range(-1, 2):
        for y in range(-1, 2):
            output += img[row+x, col+y] * kernel[x+1, y+1]

    return output


def sobel(img, kernel):
    
    padded_img = np.pad(img, pad_width=1, mode='constant', constant_values=0)

    x_dim, y_dim = padded_img.shape

    grad_x = np.zeros_like(padded_img, dtype=np.float64)
    for row in range(1, x_dim-1):
        for col in range(1, y_dim-1):
            grad_x[row, col] = manual_correlation(padded_img, row, col, kernel) 

    return grad_x[1:-1, 1:-1]

kernel_x = np.array([[1, 0, -1],[2, 0, -2],[1, 0, -1]])
kernel_y = np.array([[1, 2, 1],[0, 0, 0],[-1, -2, -1]])

grad_x = sobel(grayscale_pepper, kernel_x)
grad_y = sobel(grayscale_pepper, kernel_y) 


grad_image = cv2.normalize(cv2.magnitude(grad_x, grad_y), None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_64F)

fig, ax = plt.subplots(1, 2, figsize=(6,4))
ax[0].imshow(grayscale_pepper, cmap="gray")
ax[0].set_title("Original Image")
ax[1].imshow(grad_image, cmap="gray")
ax[1].set_title("Sobel Magnitude Image")

plt.show()

In [ ]:

########################## Task 1B ###########################

# b) Prewitt edge detector (Apply horizontal and vertical prewitt filters and display the Prewitt magnitude image)

prewitt_kernel_x = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
prewitt_kernel_y = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])

# I can repurpose "sobel" function which is actually just manual correlation, but I don't want to change the name
prewitt_x = sobel(grayscale_pepper, prewitt_kernel_x)
prewitt_y = sobel(grayscale_pepper, prewitt_kernel_y)

# grad_image = cv2.normalize(cv2.magnitude(grad_x, grad_y), None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_64F)
prewitt_magnitude = cv2.normalize(cv2.magnitude(prewitt_x, prewitt_y), None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_64F)

fig, ax = plt.subplots(1, 2, figsize=(6,4))
ax[0].imshow(grayscale_pepper, cmap="gray")
ax[0].set_title("Original Image")
ax[1].imshow(prewitt_magnitude, cmap="gray")
ax[1].set_title("Prewitt Magnitude")
plt.show()

In [ ]:

########################## Task 1C ###########################

# c) Laplacian edge detector 

laplacian_kernel = np.array([[0,-1,0], [-1,4,-1], [0,-1,0]])

laplacian_magnitude_image = cv2.normalize(sobel(grayscale_pepper, laplacian_kernel), None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_64F)

fig, ax = plt.subplots(1, 2, figsize=(6,4))
ax[0].imshow(grayscale_pepper, cmap="gray")
ax[0].set_title("Original Image")
ax[1].imshow(laplacian_magnitude_image, cmap="gray")
ax[1].set_title("Laplacian Magnitude")
plt.show()

In [ ]:

########################## Task 1D ###########################

# d) Edge detection using difference of Gaussians (DoG)

# This is a google search AI code generation to get precomputed guassian kernel as opposed to manually providing values.grayscale_pepper
def precompute_gaussian_kernel(size, sigma):
    # 1. Create a coordinate grid centered at 0
    ax = np.linspace(-(size - 1) / 2.0, (size - 1) / 2.0, size)
    x, y = np.meshgrid(ax, ax)
    
    # 2. Apply the deterministic Gaussian formula
    kernel = np.exp(-0.5 * (x**2 + y**2) / (sigma**2))
    
    # 3. Normalize so the sum of all elements equals 1
    return kernel / np.sum(kernel)

difference_kernel = precompute_gaussian_kernel(3, 1) - precompute_gaussian_kernel(3, 4)

difference_magnitude_image = cv2.normalize(sobel(grayscale_pepper, difference_kernel), None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_64F)
 
fig, ax = plt.subplots(1, 2, figsize=(6,4))
ax[0].imshow(grayscale_pepper, cmap="gray")
ax[0].set_title("Original Image")
ax[1].imshow(difference_magnitude_image, cmap="gray")
ax[1].set_title("Difference Magnitude")
plt.show()

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(6, 7))
ax[0,0].imshow(grad_image, cmap="gray")
ax[0,1].imshow(prewitt_magnitude, cmap="gray")
ax[1,0].imshow(laplacian_magnitude_image, cmap="gray")
ax[1,1].imshow(difference_magnitude_image, cmap="gray")

ax[0,0].set_title("1A Sobel")
ax[0,1].set_title("1B Prewitt")
ax[1,0].set_title("1C Laplacian")
ax[1,1].set_title("1D Difference")

plt.show()

### Task 2 (40 points)

<img src = Images/zebra.jpeg>

The Canny edge detector is an edge detection algorithm with multiple steps. The steps in the Canny edge detector are listed below: 

    - Smooth the image to remove the noise (Gaussian filter)
    - Find the gradients of the image
    - Apply non-maximum suppression
    - Apply double threshold. Select edge pixels by hysteresis (suppress all edges that are weak and not connected to strong edges).

Implement "Canny Edge Detector" from scratch using the steps above (for each step above, you may use library functions). 

In [ ]:
imported_zebra_bgr = cv2.cvtColor(cv2.imread("Images/zebra.jpeg"), cv2.COLOR_BGR2GRAY)

smoothed_guass = cv2.GaussianBlur(imported_zebra_bgr, (3,3), 0)

grad_x_zebra = cv2.Sobel(smoothed_guass, ddepth=cv2.CV_64F, dx=1, dy=0, ksize=3, scale=1, delta=1, borderType=cv2.BORDER_DEFAULT)
grad_y_zebra = cv2.Sobel(smoothed_guass, ddepth=cv2.CV_64F, dx=0, dy=1, ksize=3, scale=1, delta=1, borderType=cv2.BORDER_DEFAULT)

gradient__magnitude_zebra, theta_matrix = cv2.cartToPolar(grad_x_zebra, grad_y_zebra, angleInDegrees=True)

# I stole this from google AI search. This does pixel interpolation via binning the angles, such that there are 8 distinct directions (4 axises) 
def non_max_suppression(gradient_magnitude, gradient_direction):
    """
    Applies non-maximum suppression to thin out edges in Canny edge detection.
    
    Parameters:
    - gradient_magnitude: 2D numpy array of gradient magnitudes (intensities)
    - gradient_direction: 2D numpy array of gradient angles in radians
    
    Returns:
    - z: 2D numpy array with suppressed (thinned) edges
    """
    M, N = gradient_magnitude.shape
    z = np.zeros((M, N), dtype=np.float64)
    
    # Convert gradient direction from radians to degrees
    angle = gradient_direction % 180.0
    angle[angle < 0] += 180.0  # Map negative angles to [0, 180)

    for i in range(1, M - 1):
        for j in range(1, N - 1):
            q = 255.0
            r = 255.0

            # Angle 0 degrees (East-West direction -> compare Left and Right)
            if (0 <= angle[i, j] < 22.5) or (157.5 <= angle[i, j] <= 180):
                q = gradient_magnitude[i, j + 1]
                r = gradient_magnitude[i, j - 1]
            
            # Angle 45 degrees (North-East/South-West direction -> compare Top-Right and Bottom-Left)
            elif (22.5 <= angle[i, j] < 67.5):
                q = gradient_magnitude[i + 1, j - 1]
                r = gradient_magnitude[i - 1, j + 1]
            
            # Angle 90 degrees (North-South direction -> compare Top and Bottom)
            elif (67.5 <= angle[i, j] < 112.5):
                q = gradient_magnitude[i + 1, j]
                r = gradient_magnitude[i - 1, j]
            
            # Angle 135 degrees (North-West/South-East direction -> compare Top-Left and Bottom-Right)
            elif (112.5 <= angle[i, j] < 157.5):
                q = gradient_magnitude[i - 1, j - 1]
                r = gradient_magnitude[i + 1, j + 1]

            # Suppress if not greater than or equal to neighbors along the edge normal
            if (gradient_magnitude[i, j] >= q) and (gradient_magnitude[i, j] >= r):
                z[i, j] = gradient_magnitude[i, j]
            else:
                z[i, j] = 0.0

    return z


non_maxed_zebra = non_max_suppression(gradient__magnitude_zebra, theta_matrix) 

# I asked google AI for the manually implementation
# I was going to do the worst possible implementation of this, but after seeing the use of a queue, this is much better than what I was about to do...
def manual_hysteresis_thresholding(img, low_thresh, high_thresh):
    # 1. Initialize output and identify pixel classifications
    strong_edges = (img >= high_thresh)
    weak_edges = (img >= low_thresh) & (img < high_thresh)
    
    # Output image initialized to 0
    output = np.zeros_like(img, dtype=np.uint8)
    
    # 2. Setup the queue for tracking connectivity (BFS)
    # We start with the coordinates of all strong edges
    strong_i, strong_j = np.where(strong_edges)
    queue = list(zip(strong_i, strong_j))
    
    # Mark strong edges in the final output immediately
    output[strong_edges] = 255
    
    # Define 8-connected neighborhood offsets
    neighbor_offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1)
    ]
    
    height, width = img.shape
    
    # 3. Traverse the image using a queue (Flood Fill / BFS)
    while queue:
        curr_i, curr_j = queue.pop(0)
        
        # Check all 8 neighbors
        for di, dj in neighbor_offsets:
            ni, nj = curr_i + di, curr_j + dj
            
            # Ensure neighbor is within image boundaries
            if 0 <= ni < height and 0 <= nj < width:
                # If the neighbor is a weak edge, promote it to a strong edge
                if weak_edges[ni, nj] and output[ni, nj] == 0:
                    output[ni, nj] = 255
                    queue.append((ni, nj)) # Add to queue to check its neighbors
                    
    return output

high_threshold = 500
low_threshold = high_threshold * .9 
hysteresis_zebra = manual_hysteresis_thresholding(non_maxed_zebra, low_threshold, high_threshold)

plt.imshow(hysteresis_zebra, cmap="gray")
plt.title("Task 2")
plt.show()

### Task 3 (30 points) 

Implement Hough transform to detect lines in the image. Superimpose detected lines on the original image.

Note: You can't use cv2.HoughLines() or any other library fuctions to detect the lines in one line of code. You have to follow the voting process mentioned in the slide.

<img src = Images/hough_lines.jpg>

In [ ]:

# To be safe I will redo all the steps from Task 2 for edge detection.
imported_hough_lines = cv2.cvtColor(cv2.imread("Images/hough_lines.jpg"), cv2.COLOR_BGR2GRAY)

smoothed_hough = cv2.GaussianBlur(imported_hough_lines, (3,3), 0)

grad_x_hough = cv2.Sobel(smoothed_hough, ddepth=cv2.CV_64F, dx=1, dy=0, ksize=3, scale=1, delta=1, borderType=cv2.BORDER_DEFAULT)
grad_y_hough = cv2.Sobel(smoothed_hough, ddepth=cv2.CV_64F, dx=0, dy=1, ksize=3, scale=1, delta=1, borderType=cv2.BORDER_DEFAULT)

gradient_magnitude_hough, theta_matrix = cv2.cartToPolar(grad_x_hough, grad_y_hough, angleInDegrees=True)

non_maxed_hough = non_max_suppression(gradient_magnitude_hough, theta_matrix) 

high_threshold = 40
low_threshold = high_threshold * .90 
hysteresis_hough = manual_hysteresis_thresholding(non_maxed_hough, low_threshold, high_threshold)

# The first step is achieved
#plt.imshow(hysteresis_hough, cmap="gray")

In [ ]:

# Now the next step is for us to do make the accumulator array and do all those
# steps. But first I need to figure out how to make sin waves so its time to
# look at the notes.
p_max = int(np.ceil(np.hypot(imported_hough_lines.shape[1], imported_hough_lines.shape[0])))


theta_accumulator = 180 
p_accumulator = 2 * p_max + 1 # This is not in the slides but this is an implementation choice to ensure the offset p values do not exceed the range: This is because we want to offset the negetaive p values to make them positive.
H = np.zeros((theta_accumulator, p_accumulator), dtype=int)

edges = np.nonzero(hysteresis_hough)

for (x,y) in zip(*edges):
    for theta in range(0, 180):
        p = x * np.cos(np.radians(theta)) + y * np.sin(np.radians(theta))

        p_index = int(np.round(p)) + p_max # This is the offset to ensure there are no negatives

        H[theta, p_index] = H[theta, p_index] + 1




plt.imshow(H, cmap="gray")

In [ ]:

# Finally to find local maximum: We will use threshold instead since I am lazy
line_threshold = 125
passed_lines = np.argwhere(H >= line_threshold)


plt.figure(figsize=(10, 6))
plt.imshow(H, cmap="gray", aspect="auto")

plt.scatter(
    passed_lines[:, 1],  # rho indices (x-axis)
    passed_lines[:, 0],  # theta indices (y-axis)
    color="red",
    s=10
)

plt.xlabel("Rho index")
plt.ylabel("Theta (degrees)")
plt.title("Detected Hough Peaks")
plt.show()

In [ ]:

# Just kidding the last step is to map back into line equations.
lines = []

for (theta, p) in passed_lines:
    p = p - p_max # undo offset
    theta = np.radians(theta)

    x0 = p * np.cos(theta)
    y0 = p * np.sin(theta)

    length = int(np.hypot(imported_hough_lines.shape[0], imported_hough_lines.shape[1])) # ChatGPT uses this length to conveniently get the line points

    # Use these initial x0 and y0, then we can use the normal form equation to give us x and y values to plot. Note that these are derived versions that use tri identities
    x1 = int(round(x0 - length * np.sin(theta)))
    y1 = int(round(y0 + length * np.cos(theta))) 

    x2 = int(round(x0 + length * np.sin(theta)))
    y2 = int(round(y0 - length * np.cos(theta))) 
    lines.append(((x1, y1), (x2, y2)))

hough_output = imported_hough_lines.copy()

for point1, point2 in lines:
    cv2.line(hough_output, point1, point2, color=255, thickness=2)

plt.imshow(hough_output)

## Submission Guidelines:

    Submit through Canvas your source code in a single .ipynb file. The name of the .ipynb file should be YourStudentID.ipynb. (For example: 1001234567.ipynb)
    Import all the images from the ./Images directory. Your TA will use the same directory name to grade your submission.
    You don't need to attach the image file with your submission.